In [89]:
# Cell 1: Database Connection
import sqlite3
import pandas as pd
import numpy as np
import math
import json
import time

conn   = sqlite3.connect("urban_ai.db")
cursor = conn.cursor()

print("Connected to urban_ai.db")

Connected to urban_ai.db


In [90]:
def wgs84_to_utm_zone19n_nad83(latitude, longitude):
    a  = 6_378_137.0
    f  = 1 / 298.257222101
    e2 = 2 * f - f ** 2
    k0 = 0.9996
    E0 = 500_000.0

    lon0 = math.radians(-69.0)
    latitude_r  = math.radians(latitude)
    longitude_r = math.radians(longitude)

    N = a / math.sqrt(1 - e2 * math.sin(latitude_r) ** 2)
    T = math.tan(latitude_r) ** 2
    C = (e2 / (1 - e2)) * math.cos(latitude_r) ** 2
    A = math.cos(latitude_r) * (longitude_r - lon0)

    M = a * (
        (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256) * latitude_r
        - (3*e2/8 + 3*e2**2/32 + 45*e2**3/1024) * math.sin(2 * latitude_r)
        + (15*e2**2/256 + 45*e2**3/1024) * math.sin(4 * latitude_r)
        - (35*e2**3/3072) * math.sin(6 * latitude_r)
    )

    x = E0 + k0 * N * (
        A
        + (1 - T + C) * A**3 / 6
        + (5 - 18*T + T**2 + 72*C - 58*(e2/(1-e2))) * A**5 / 120
    )

    y = k0 * (
        M + N * math.tan(latitude_r) * (
            A**2 / 2
            + (5 - T + 9*C + 4*C**2) * A**4 / 24
            + (61 - 58*T + T**2 + 600*C - 330*(e2/(1-e2))) * A**6 / 720
        )
    )

    return x, y

print("UTM projection function ready.")

UTM projection function ready.


In [91]:
# ==============================================================
# Data Initialization
# ==============================================================
cbgs = pd.read_csv("worcester_cbgs.csv")
pois = pd.read_csv("worcester_pois.csv")
visits = pd.read_csv("worcester_cbg_poi_visits.csv")
distances = pd.read_csv("worcester_cbg_poi_distance.csv")
parameters = pd.read_csv("calibrated_parameters_filtered.csv")

with open("worcester_cbgs_map.geojson", "r") as f:
    geojson = json.load(f)
 
centroid_rows = []
for feature in geojson["features"]:
    props = feature["properties"]
    centroid_rows.append({
        "cbg": int(props["GEOID10"]),
        "lat": float(props["INTPTLAT10"]),
        "lon": float(props["INTPTLON10"])
    })
 
cbg_centroids = pd.DataFrame(centroid_rows)
cbgs = cbgs.merge(cbg_centroids, on="cbg", how="left")

print("Files loaded:")
print(f"  CBGs:       {len(cbgs)} neighborhoods")
print(f"  POIs:       {len(pois)} existing businesses")
print(f"  Visits:     {len(visits)} records")
print(f"  Distances:  {len(distances)} records")
print(f"  Categories: {len(parameters)} parameter sets")

Files loaded:
  CBGs:       149 neighborhoods
  POIs:       4069 existing businesses
  Visits:     26924 records
  Distances:  606281 records
  Categories: 23 parameter sets


In [92]:
# Create Tables

cursor.executescript("""

-- Table 1: CBG Master (demographics + coordinates + UTM)
CREATE TABLE IF NOT EXISTS cbg_master (
    cbg_id          INTEGER PRIMARY KEY,
    population      REAL,
    median_income   REAL,
    median_age      REAL,
    latitude        REAL,
    longitude       REAL,
    utm_x           REAL,
    utm_y           REAL
);

-- Table 2: POIs (existing businesses)
CREATE TABLE IF NOT EXISTS pois (
    placekey        TEXT PRIMARY KEY,
    top_category    TEXT,
    naics_code      INTEGER,
    area_sq_meters  REAL,
    utm_x           REAL,
    utm_y           REAL
);

-- Table 3: CBG-POI Stats (distance + visits)
CREATE TABLE IF NOT EXISTS cbg_poi_stats (
    cbg_id          INTEGER,
    placekey        TEXT,
    top_category    TEXT,
    naics_code      INTEGER,
    distance_m      REAL,
    visit_count     INTEGER DEFAULT 0,
    PRIMARY KEY (cbg_id, placekey)
);

""")

conn.commit()
print("Tables created successfully.")

Tables created successfully.


In [93]:
# Insert data into cbg_master

print("Inserting into cbg_master")

for _, row in cbgs.iterrows():
    utm_x, utm_y = wgs84_to_utm_zone19n_nad83(row["lat"], row["lon"])
    
    cursor.execute("""
        INSERT OR REPLACE INTO cbg_master
        (cbg_id, population, median_income, median_age, latitude, longitude, utm_x, utm_y)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        int(row["cbg"]),
        row["total_population"],
        row["median_household_income"],
        row["median_age"],
        row["lat"],
        row["lon"],
        utm_x,
        utm_y
    ))

conn.commit()
print(f"cbg_master: {len(cbgs)} rows inserted.")

Inserting into cbg_master
cbg_master: 149 rows inserted.


In [94]:
# Insert data into pois

print("Inserting into pois")

for _, row in pois.iterrows():
    utm_x, utm_y = wgs84_to_utm_zone19n_nad83(row["latitude"], row["longitude"])
    
    cursor.execute("""
        INSERT OR REPLACE INTO pois
        (placekey, top_category, naics_code, area_sq_meters, utm_x, utm_y)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row["placekey"],
        row["top_category"],
        row["naics_code"],
        row["wkt_area_sq_meters"],
        utm_x,
        utm_y
    ))

conn.commit()
print(f"pois: {len(pois)} rows inserted.")

Inserting into pois
pois: 4069 rows inserted.


In [95]:
# Insert data into cbg_poi_stats
# Merge distances (606,281 rows) with visits (26,924 rows)
# visits not in distances will get visit_count = 0

print("Merging distances and visits")

# Rename columns to match our schema
distances_clean = distances.rename(columns={"GEOID10": "cbg_id"})
visits_clean = visits.rename(columns={"visitor_home_cbg": "cbg_id"})

# Left join: distances is the base, visits fills in visit_count
merged = distances_clean.merge(
    visits_clean[["cbg_id", "placekey", "visit_count"]],
    on=["cbg_id", "placekey"],
    how="left"
)

# Fill missing visit_count with 0
merged["visit_count"] = merged["visit_count"].fillna(0).astype(int)

# Bring in top_category and naics_code from pois
merged = merged.merge(
    pois[["placekey", "top_category", "naics_code"]],
    on="placekey",
    how="left"
)

print(f"Merged table: {len(merged)} rows")
print("Inserting into cbg_poi_stats...")

merged.to_sql(
    "cbg_poi_stats",
    conn,
    if_exists="replace",
    index=False
)

conn.commit()
print(f"cbg_poi_stats: {len(merged)} rows inserted.")

Merging distances and visits
Merged table: 606281 rows
Inserting into cbg_poi_stats...
cbg_poi_stats: 606281 rows inserted.


In [96]:
# Pre-compute competitor utility per CBG per category

print("Pre-computing competitor utilities")

# Get all unique categories
categories = parameters[["top_category", "NAICS code", "alpha", "beta"]].drop_duplicates()

results = []

for _, cat_row in categories.iterrows():
    top_category = cat_row["top_category"]
    naics_code   = cat_row["NAICS code"]
    alpha        = cat_row["alpha"]
    beta         = cat_row["beta"]

    # Get all competitors in this category
    competitors = pois[
        (pois["top_category"] == top_category) |
        (pois["naics_code"].astype(str) == str(naics_code))
    ]

    if len(competitors) == 0:
        continue

    # Get distances for these competitors
    comp_stats = distances[
        distances["placekey"].isin(competitors["placekey"])
    ].copy()

    comp_stats = comp_stats.merge(
        competitors[["placekey", "wkt_area_sq_meters"]],
        on="placekey",
        how="left"
    )

    median_size = competitors["wkt_area_sq_meters"].median()
    comp_stats["wkt_area_sq_meters"] = comp_stats["wkt_area_sq_meters"].fillna(median_size)
    comp_stats["distance_m"] = comp_stats["distance_m"].replace(0, 1.0)

    # Calculate utility for each competitor
    comp_stats["U_comp"] = (
        comp_stats["wkt_area_sq_meters"] ** alpha /
        comp_stats["distance_m"] ** beta
    )

    # Sum per CBG
    cbg_sum = (
        comp_stats.groupby("GEOID10")["U_comp"]
        .sum()
        .reset_index()
        .rename(columns={"GEOID10": "cbg_id", "U_comp": "sum_U_existing"})
    )

    cbg_sum["top_category"] = top_category
    cbg_sum["naics_code"]   = naics_code
    results.append(cbg_sum)

# Combine all categories
competitor_utility = pd.concat(results, ignore_index=True)

# Save to database
competitor_utility.to_sql(
    "competitor_utility",
    conn,
    if_exists="replace",
    index=False
)

conn.commit()
print(f"competitor_utility: {len(competitor_utility)} rows inserted.")
print(f"Categories covered: {competitor_utility['top_category'].nunique()}")

Pre-computing competitor utilities
competitor_utility: 3427 rows inserted.
Categories covered: 23


In [97]:
# Add params table to database

parameters.rename(columns={"NAICS code": "naics_code"}).to_sql(
    "params",
    conn,
    if_exists="replace",
    index=False
)

conn.commit()
print("params table inserted.")

params table inserted.


In [98]:
# Close connection
conn.commit()
conn.close()
print("Database saved and closed.")

Database saved and closed.
